In [426]:
import folium
from folium import plugins
import webbrowser
import pandas as pd
import numpy as np
import math

In [427]:
# Set filters
# Set to a number or NONE for no filter
MIN_GROUP_SIZE = 30 
DAY_TYPE = 0 #(0 = weekday, 1 = Saturday, 2 = Sunday)
START_HOUR = None

In [428]:
# Load in data and filter
data = pd.read_csv("../data/sabrina_500_merged.csv", sep=",")
if MIN_GROUP_SIZE is not None:
    data = data[(data["group_n"]>= MIN_GROUP_SIZE)]

if DAY_TYPE is not None:
    data = data[data["day_type"] == DAY_TYPE]

if START_HOUR is not None:
    data = data[data["start_hour"] == START_HOUR]

# Define column names for convenience
pickup_lat = "Pickup Centroid Latitude"
pickup_long = "Pickup Centroid Longitude"
dropoff_lat = "Dropoff Centroid Latitude"
dropoff_long = "Dropoff Centroid Longitude"

# Adding columns for later reformatting:
# - pickup_point: (lat, long) tuple
# - dropoff_point: (lat, long) tuple
# - points: this is a "frozen set" of {pickup_point, dropoff_point}, it is an unordered
# but hashable type of set that we can use as to group sets of points
data["pickup_point"] = data.apply(lambda row: (row[pickup_lat],row[pickup_long]), axis = 1)
data["dropoff_point"] = data.apply(lambda row: (row[dropoff_lat],row[dropoff_long]), axis=1)
data["points"] = data.apply(lambda row: [row["pickup_point"], row["dropoff_point"]], axis=1).apply(frozenset)


def standardize(x):
    """
    Standardizes value, used as weighting function for opacity/size
    """
    return (x-x.min())/(x.max()-x.min())

data.sort_values("group_n")

,day_type,start_hour,Pickup Centroid Latitude,Pickup Centroid Longitude,Dropoff Centroid Latitude,Dropoff Centroid Longitude,group_n,group_id,representative_year,representative_month,...,walkDist,walkTime,transitDist,transitTime,totalDist,totalTime,modes,pickup_point,dropoff_point,points
427,0,20,41.861,-87.631,41.893,-87.626,31,1774649,2026,4,...,817.0,735.0,3653.0,1234.0,4470.0,1970s,"{'BUS', 'WALK'}","(41.861, -87.631)","(41.893, -87.626)","frozenset({(41.893, -87.626), (41.861, -87.631)})"
422,0,21,41.877,-87.681,41.893,-87.638,32,1933438,2026,4,...,207.0,178.0,5614.0,1876.0,5821.0,2576s,"{'BUS', 'WALK'}","(41.877, -87.681)","(41.893, -87.638)","frozenset({(41.877, -87.681), (41.893, -87.638)})"
108,0,13,41.963,-87.713,41.920,-87.761,32,951425,2026,4,...,511.0,468.0,8377.0,1972.0,8888.0,2786s,"{'BUS', 'WALK'}","(41.963, -87.713)","(41.92, -87.761)","frozenset({(41.963, -87.713), (41.92, -87.761)})"
432,0,23,41.958,-87.660,41.949,-87.662,33,2312639,2026,4,...,986.0,850.0,673.0,243.0,1659.0,1093s,"{'BUS', 'WALK'}","(41.958, -87.66)","(41.949, -87.662)","frozenset({(41.958, -87.66), (41.949, -87.662)})"
180,0,13,41.893,-87.638,41.935,-87.640,33,915016,2026,4,...,351.0,330.0,6517.0,1543.0,6868.0,2086s,"{'BUS', 'WALK'}","(41.893, -87.638)","(41.935, -87.64)","frozenset({(41.935, -87.64), (41.893, -87.638)})"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,0,22,41.881,-87.633,41.979,-87.903,6216,2090674,2026,4,...,1733.0,1554.0,28445.0,2820.0,30178.0,4375s,"{'SUBWAY', 'WALK'}","(41.881, -87.633)","(41.979, -87.903)","frozenset({(41.979, -87.903), (41.881, -87.633)})"
155,0,18,41.979,-87.903,41.885,-87.621,6402,1573550,2026,4,...,2227.0,1912.0,28148.0,2550.0,30375.0,4462s,"{'SUBWAY', 'WALK'}","(41.979, -87.903)","(41.885, -87.621)","frozenset({(41.979, -87.903), (41.885, -87.621)})"
352,0,19,41.979,-87.903,41.885,-87.621,6743,1709554,2026,4,...,2227.0,1912.0,28148.0,2550.0,30375.0,4462s,"{'SUBWAY', 'WALK'}","(41.979, -87.903)","(41.885, -87.621)","frozenset({(41.979, -87.903), (41.885, -87.621)})"
27,0,18,41.885,-87.621,41.979,-87.903,6770,1518416,2026,4,...,1549.0,1331.0,29126.0,2909.0,30675.0,4531s,"{'SUBWAY', 'BUS', 'WALK'}","(41.885, -87.621)","(41.979, -87.903)","frozenset({(41.979, -87.903), (41.885, -87.621)})"


In [429]:
# Create point sets data frame
# This is so that we can draw bidirectional routes
point_sets =pd.DataFrame(data.groupby(by=["points","pickup_point", "dropoff_point"])["group_n"].sum())
point_sets['line_weight'] = (standardize(point_sets['group_n']) + 0.1) * 5
point_sets['opacity'] = standardize(point_sets['group_n']) + 0.1
# point_sets.sort_values("group_n")

In [430]:
# Create dataset of nodes
# Pickup and dropoff locations are often the same, so condense it so that
# we draw each node only once
pickup_nodes = data[[pickup_lat, pickup_long, "group_n"]]
pickup_nodes.columns = ["latitude", "longitude", "group_n"]
dropoff_nodes = data[[dropoff_lat, dropoff_long, "group_n"]]
dropoff_nodes.columns = ["latitude", "longitude", "group_n"]
all_nodes = pd.concat([pickup_nodes, dropoff_nodes], axis = 0)

# Radius and opacity are based on how many rides start OR end at that spot
all_nodes = all_nodes.groupby(["latitude", "longitude"])["group_n"].agg(sum).reset_index()
all_nodes["radius"] = (standardize(all_nodes["group_n"]) + 0.1) * 12
all_nodes["opacity"] = (standardize(all_nodes["group_n"]) + 0.1) 
# all_nodes.sort_values(by = "group_n")


In [431]:
# Make the Map
map = folium.Map(
    location=(41.86721, -87.63231),
    zoom_control=False,
    tiles="Cartodb Positron", 
    zoom_start=12
)

# Plot map nodes
for i, node in all_nodes.iterrows():
    folium.CircleMarker(
        location=[node["latitude"], node["longitude"]],
        radius=node["radius"],
        color="cornflowerblue",
        stroke=False,
        fill=True,
        fill_opacity=node["opacity"],
        tooltip= f"Node: {int(node["group_n"])} rides",
        opacity=1).add_to(map)

# Plot ap routes
for index in point_sets.index.get_level_values(0):
    subset = point_sets[point_sets.index.get_level_values(0).isin([index])]
    for index, value in subset.iterrows():
        route = [index[1], index[2]]
        
        line = folium.plugins.PolyLineOffset(route, 
                                             color="cornflowerblue", 
                                             opacity=value["opacity"],
                                             weight = value["line_weight"], 
                                             offset=-5,
                                             tooltip= f"Route: {int(value["group_n"])} rides").add_to(map)
        # Add arrows
        folium.plugins.PolyLineTextPath(line,
                                        text="\u2192",
                                        repeat=True,
                                        center = True,
                                        offset = 9,
                                        attributes={"font-size": "30", 
                                                    "fill" : "cornflowerblue",
                                                    "fill-opacity": str(value["opacity"])}).add_to(map)

    
map.save("map.html")
webbrowser.open("map.html")
map.show_in_browser()

Your map should have been opened in your browser automatically.
Press ctrl+c to return.
